# FFA-Net
Qin et al. (AAAI 2020). Re-implemented below with the same layer names as the official repo (zhilin007/FFA-Net), so the official pretrained weights load directly. Use a GPU runtime.

In [ ]:
import numpy as np
import torch
import torch.nn as nn
import torchvision.transforms as tfs

# FFA-Net (Qin et al., AAAI 2020) is re-implemented in PyTorch below, same architecture as the
# official repo (zhilin007/FFA-Net), so there is no repo to clone. Only the pretrained .pk weights are needed.
print(torch.__version__, "| CUDA:", torch.cuda.is_available())

## Data

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

DRIVE     = "/content/drive/MyDrive"
SRC       = f"{DRIVE}/O-HAZE"          # original O-HAZE: hazy/ + GT/
DATA_ROOT = f"{DRIVE}/O-HAZE-1600"     # resized copy, shared by all methods and the evaluation
OUT       = f"{DRIVE}/outputs/ffa_net" # dehazed images, same file names as hazy/

MAX_IMAGES = 5             # first N pairs (None = all 45)
RESIZE = 1600              # long side; O-HAZE is ~3000x4000

In [ ]:
#resize hazy + GT once (skipped if DATA_ROOT already exists), same resizing as before:
#hazy: long side -> RESIZE (bicubic), GT: resized to the hazy size
import os, glob
from PIL import Image

def resize_long(img, long_side):
    if long_side <= 0 or max(img.size) <= long_side:
        return img
    s = long_side / max(img.size)
    return img.resize((round(img.size[0] * s), round(img.size[1] * s)), Image.BICUBIC)

if not os.path.isdir(f"{DATA_ROOT}/hazy"):
    os.makedirs(f"{DATA_ROOT}/hazy"); os.makedirs(f"{DATA_ROOT}/GT")
    pairs = list(zip(sorted(glob.glob(f"{SRC}/hazy/*")), sorted(glob.glob(f"{SRC}/GT/*"))))[:MAX_IMAGES]
    for h, g in pairs:
        hazy = resize_long(Image.open(h).convert('RGB'), RESIZE)
        gt   = Image.open(g).convert('RGB').resize(hazy.size, Image.BICUBIC)
        stem = lambda p: os.path.splitext(os.path.basename(p))[0]
        hazy.save(f"{DATA_ROOT}/hazy/{stem(h)}.png"); gt.save(f"{DATA_ROOT}/GT/{stem(g)}.png")

hazy_paths = sorted(glob.glob(f"{DATA_ROOT}/hazy/*"))
os.makedirs(OUT, exist_ok=True)
print(len(hazy_paths), "images:", [os.path.basename(p) for p in hazy_paths])

## Pretrained weights (official FFA)

In [ ]:
!pip -q install gdown
!gdown --folder "https://drive.google.com/drive/folders/19_lSUPrpLDZl9AyewhHBsHidZEpTMIV5" -O /content/trained_models
!ls -lh /content/trained_models

## Parameters

In [ ]:
# FFA settings -- defaults follow the official repo (zhilin007/FFA-Net)
FFA_GPS = 3                # number of groups (official: 3)
FFA_BLOCKS = 19            # blocks per group (official: 19)
NORM_MEAN = [0.64, 0.6, 0.58]     # input normalization from official test.py
NORM_STD  = [0.14, 0.15, 0.152]   # do NOT change, pretrained weights expect this distribution
WEIGHTS = '/content/trained_models/ots_train_ffa_3_19.pk'   # OTS = outdoor, closer to O-HAZE than ITS (indoor)

# inference settings
TILE = 512                 # sliding-window size; full image would OOM
OVERLAP = 64               # overlap between tiles, blended with feathering to hide seams
AMP = True                 # fp16 autocast on GPU, faster and uses less memory

DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(DEVICE)

## Model

In [ ]:
# FFA-Net (Qin et al., AAAI 2020) -- same architecture and layer names as the official net/models/FFA.py,
# so the pretrained state_dict loads with strict=True.

def default_conv(in_ch, out_ch, k, bias=True):
    # 'same' padding: output keeps the input's spatial size
    return nn.Conv2d(in_ch, out_ch, k, padding=k // 2, bias=bias)


class PALayer(nn.Module):
    """Pixel Attention: one weight per pixel, since haze is spatially non-uniform."""
    def __init__(self, c):
        super().__init__()
        self.pa = nn.Sequential(
            nn.Conv2d(c, c // 8, 1), nn.ReLU(inplace=True),
            nn.Conv2d(c // 8, 1, 1), nn.Sigmoid())       # -> (B, 1, H, W) map in [0, 1]

    def forward(self, x):
        return x * self.pa(x)


class CALayer(nn.Module):
    """Channel Attention: one weight per feature channel."""
    def __init__(self, c):
        super().__init__()
        self.avg_pool = nn.AdaptiveAvgPool2d(1)          # global average -> (B, C, 1, 1)
        self.ca = nn.Sequential(
            nn.Conv2d(c, c // 8, 1), nn.ReLU(inplace=True),
            nn.Conv2d(c // 8, c, 1), nn.Sigmoid())       # -> (B, C, 1, 1) weights in [0, 1]

    def forward(self, x):
        return x * self.ca(self.avg_pool(x))


class Block(nn.Module):
    """Basic block: conv -> ReLU (+ local residual) -> conv -> CA -> PA, with an outer residual."""
    def __init__(self, conv, dim, k):
        super().__init__()
        self.conv1, self.act1 = conv(dim, dim, k), nn.ReLU(inplace=True)
        self.conv2 = conv(dim, dim, k)
        self.calayer, self.palayer = CALayer(dim), PALayer(dim)

    def forward(self, x):
        res = self.act1(self.conv1(x)) + x               # local residual learning
        res = self.palayer(self.calayer(self.conv2(res)))
        return res + x


class Group(nn.Module):
    """Group: `blocks` Blocks followed by one conv, with a residual around the whole group."""
    def __init__(self, conv, dim, k, blocks):
        super().__init__()
        self.gp = nn.Sequential(*[Block(conv, dim, k) for _ in range(blocks)], conv(dim, dim, k))

    def forward(self, x):
        return self.gp(x) + x


class FFA(nn.Module):
    """pre conv -> 3 Groups -> Feature Attention fusion -> PA -> post convs, plus a global residual."""
    def __init__(self, gps=3, blocks=19, conv=default_conv):
        super().__init__()
        self.gps, self.dim, k = gps, 64, 3               # 64 feature channels, 3x3 kernels
        self.g1 = Group(conv, self.dim, k, blocks)
        self.g2 = Group(conv, self.dim, k, blocks)
        self.g3 = Group(conv, self.dim, k, blocks)

        # Feature Attention: learns how much each group's output (and each channel in it) contributes
        self.ca = nn.Sequential(
            nn.AdaptiveAvgPool2d(1),
            nn.Conv2d(self.dim * gps, self.dim // 16, 1), nn.ReLU(inplace=True),
            nn.Conv2d(self.dim // 16, self.dim * gps, 1), nn.Sigmoid())
        self.palayer = PALayer(self.dim)

        self.pre  = nn.Sequential(conv(3, self.dim, k))                               # RGB -> 64 features
        self.post = nn.Sequential(conv(self.dim, self.dim, k), conv(self.dim, 3, k))  # 64 features -> RGB

    def forward(self, x1):
        x = self.pre(x1)
        r1 = self.g1(x)                                  # low-level features
        r2 = self.g2(r1)
        r3 = self.g3(r2)                                 # high-level features

        # weights: (B, 3*64, 1, 1) -> (B, 3, 64, 1, 1), one 64-dim weight vector per group
        w = self.ca(torch.cat([r1, r2, r3], dim=1)).view(-1, self.gps, self.dim)[:, :, :, None, None]
        out = w[:, 0] * r1 + w[:, 1] * r2 + w[:, 2] * r3  # weighted fusion of the 3 groups
        out = self.palayer(out)

        return self.post(out) + x1                       # global residual: predict the correction to the hazy input

In [ ]:
net = FFA(gps=FFA_GPS, blocks=FFA_BLOCKS)
ckpt = torch.load(WEIGHTS, map_location='cpu', weights_only=False)
state = {k.replace('module.', '', 1): v for k, v in ckpt['model'].items()}   # official checkpoint was saved from DataParallel

net.load_state_dict(state, strict=True)
net = net.to(DEVICE).eval()

print(f"FFA loaded: {sum(p.numel() for p in net.parameters())/1e6:.2f}M params, trained {ckpt.get('step', '?')} steps")

## Inference

In [ ]:
NORM = tfs.Normalize(mean=NORM_MEAN, std=NORM_STD)

@torch.no_grad()
def infer_patch(x):
    with torch.autocast(device_type=DEVICE.type, dtype=torch.float16, enabled=AMP and DEVICE.type == 'cuda'):
        return net(x).float()

@torch.no_grad()
def infer_tiled(x):
    """Sliding window + feathered blending, so tile seams are not visible."""
    _, _, H, W = x.shape
    stride = TILE - OVERLAP
    out  = torch.zeros_like(x)
    wsum = torch.zeros(1, 1, H, W, device=x.device)

    ramp = torch.ones(TILE)
    r = torch.linspace(0, 1, OVERLAP + 2)[1:-1]
    ramp[:OVERLAP], ramp[-OVERLAP:] = r, r.flip(0)
    win = (ramp[:, None] * ramp[None, :]).to(x.device)     # weight 1 in the centre, fading towards the edges

    ys = list(range(0, H - TILE + 1, stride)); ys += [H - TILE] if ys[-1] + TILE < H else []
    xs = list(range(0, W - TILE + 1, stride)); xs += [W - TILE] if xs[-1] + TILE < W else []
    for y in ys:
        for x0 in xs:
            pred = infer_patch(x[:, :, y:y + TILE, x0:x0 + TILE])
            out[:, :, y:y + TILE, x0:x0 + TILE]  += pred * win
            wsum[:, :, y:y + TILE, x0:x0 + TILE] += win
    return out / wsum

def to_uint8(t):
    return (t.clamp(0, 1)[0].permute(1, 2, 0).cpu().numpy() * 255 + 0.5).astype(np.uint8)

def dehaze(hazy):
    """hazy: PIL RGB image -> uint8 RGB array."""
    x = NORM(tfs.ToTensor()(hazy))[None].to(DEVICE)
    pred = to_uint8(infer_tiled(x))
    if DEVICE.type == 'cuda':
        torch.cuda.synchronize()
    return pred

## Run over the dataset

In [ ]:
import time
for p in hazy_paths:
    hazy = Image.open(p).convert('RGB')
    t0 = time.time()
    pred = dehaze(hazy)
    dt = time.time() - t0
    Image.fromarray(pred).save(f"{OUT}/{os.path.basename(p)}")
    print(f"{os.path.basename(p)} | {hazy.size[0]}x{hazy.size[1]} | {dt:.1f}s")

## Preview

In [ ]:
import matplotlib.pyplot as plt
for p in hazy_paths[:1]:
    fig, ax = plt.subplots(1, 2, figsize=(12, 5))
    for a, im, t in zip(ax, [Image.open(p), Image.open(f"{OUT}/{os.path.basename(p)}")], ['hazy', 'FFA']):
        a.imshow(im); a.set_title(t); a.axis('off')
    plt.tight_layout(); plt.show()

## Evaluate
Score the outputs with `dehazing/evaluation` (see `dehaze_eval_colab.ipynb`):
```python
per_image, summary = evaluate(DATA_ROOT, {"ffa_net": OUT})
```